# C-36 · PII redaction
**Requirement:** detect and mask PII before it reaches the model and before the answer reaches the user.

**Three places.**
1. **Before the model** – `guard.mask_pii()` rewrites the prompt; the model only sees `[EMAIL]`, `[CARD]`…
2. **Before the user** – the model's answer (which may quote tool results such as customer contacts) is masked on the way out.
3. **In the log** – `guard.events` stores the masked prompt, so the audit trail does not become a PII store.

Detected: e-mail addresses, card numbers, US SSNs, phone numbers. They are regular expressions in SQL (`guard.pii_types`, `guard.mask_pii`), so you can read and extend them.

In [1]:
import lab
lab.install()   # creates the guard schema, sample PII, tools, agents and purposes (safe to run again)

guard schema, sample contacts and personas: ready
tools: g_list_tickets, g_list_contacts, g_search_articles, g_assign_ticket, g_list_tickets_rls, g_ticket_notes
purposes: guard_readonly (guard_ro), guard_triage (guard_rw)
agents: guard_reader, guard_triager, guard_persona
intent examples registered for the agents


## 1 · The masking functions

In [2]:
samples = [
    "Contact jan.devries@aurorabank.example about the outage",
    "My card is 4111 1111 1111 1111, please refund",
    "Call me on +31 6 12345678 tomorrow",
    "SSN 123-45-6789 is on the form",
    "Ticket 1001 is critical",
]
rows = [(s, lab.scalar("SELECT guard.pii_types(%s)", (s,)), lab.scalar("SELECT guard.mask_pii(%s)", (s,))) for s in samples]
lab.show(rows, ["input", "pii found", "masked"], width=70)

input,pii found,masked
Contact jan.devries@aurorabank.example about the outage,"[""email""]",Contact [EMAIL] about the outage
"My card is 4111 1111 1111 1111, please refund","[""card""]","My card is [CARD], please refund"
Call me on +31 6 12345678 tomorrow,"[""phone""]",Call me on [PHONE] tomorrow
SSN 123-45-6789 is on the form,"[""ssn""]",SSN [SSN] is on the form
Ticket 1001 is critical,[],Ticket 1001 is critical


## 2 · Before the model: PII in the prompt

In [3]:
r = lab.ask("guard_reader", "My card is 4111 1111 1111 1111 and my email is jan@example.nl. Which tickets are critical?")

[ALLOWED] guard_reader <- 'My card is [CARD] and my email is [EMAIL]. Which tickets are critical?'
  intent : similarity 0.741
  pii    : card, email (masked)
  answer : Summary of actions and findings:

- The user provided their card number [CARD] and email address [EMAIL].
- The task is to identify critical tickets.
- A tool call was made to list tickets with any assignee, critical priority, and any status.
- The same result has been returned multiple times: two critical tickets are identified:
  - Ticket ID: 1001, assigned to maria, from customer Aurora Bank. Subject: "WAL volume at 97% on primary, writes failing". Tier: platinum.
  - Ticket ID: 1002, assigned to jonas, from customer Delta Health. Subject: "Need PITR to 09:15 after bad deployment". Tier: pl
  source : g_list_tickets({"assignee": "any", "priority": "critical", "status": "any"}) -> [{"id": 1001}, {"id": 1002}]
  source : g_list_tickets({"assignee": "any", "priority": "critical", "status": "any"}) -> [{"id": 1001}, {"i

In [4]:
lab.sql("SELECT id, prompt, pii_found FROM guard.events ORDER BY id DESC LIMIT 1", role="demo_governor", width=120)

id,prompt,pii_found
30,My card is [CARD] and my email is [EMAIL]. Which tickets are critical?,"[""card"", ""email""]"


The stored prompt is the masked one. The raw card number is nowhere in the database.

## 3 · Before the user: PII in the answer
The contacts table holds names, e-mail addresses and phone numbers. The tool returns them; the answer must not.

In [5]:
print("Tool result straight from the table (what the model sees):")
lab.sql("SELECT customer, contact, email, phone FROM (SELECT c.name AS customer, k.name AS contact, k.email, k.phone FROM support.contacts k JOIN support.customers c ON c.id = k.customer_id) x WHERE customer ILIKE '%Aurora%'")
print()
print("Through the request path, redaction ON:")
on = lab.ask("guard_reader", "What are the contact details of Aurora Bank?")

Tool result straight from the table (what the model sees):


customer,contact,email,phone
Aurora Bank,Tom Becker,tom.becker@aurorabank.example,+31 6 12345711



Through the request path, redaction ON:


[ALLOWED] guard_reader <- 'What are the contact details of Aurora Bank?'
  intent : similarity 0.953
  pii    : email, phone (masked)
  answer : {"contacts": [{"name": "Tom Becker", "email": "[EMAIL]", "phone": "[PHONE]"}]}
  source : g_list_contacts({"customer": "Aurora Bank"}) -> [{"id": 1}]


## 4 · Compare with redaction off

In [6]:
try:
    lab.switch("pii_redaction", enabled=False)
    print("Redaction OFF:")
    off = lab.ask("guard_reader", "What are the contact details of Aurora Bank?")
finally:
    lab.switch("pii_redaction", enabled=True)

Redaction OFF:


[ALLOWED] guard_reader <- 'What are the contact details of Aurora Bank?'
  intent : similarity 0.953
  answer : {"tool Calls": [{"name": "g_list_contacts", "arguments": {"customer": "Aurora Bank"}}]}


## What to look for
* With redaction on, the answer has `[EMAIL]` and `[PHONE]` where the contact details were; with it off they appear in clear.
* `pii_found` in the log lists the *types* only, never the values.
* Masking **in front of the model** means a hosted model provider never receives the card number or the address.
* The model still sees tool results (they come from the database, inside your boundary). To keep PII from the model altogether, mask in the tool's SQL or give the agent a view with masked columns, and use C-32's column grants.
* Regex detection has limits: names and street addresses are not caught. A NER model can be added as another step in the request path.